In [1]:
import glob
import math
import ROOT
import numpy as np
ROOT.EnableImplicitMT()
ROOT.gStyle.SetPalette(ROOT.kRainbow)
%jsroot

In [24]:
# Pobierz wszystkie pliki pasujące do wzorca
all_files = glob.glob(
    "/workspace/root_files/cnaf/2026-09-06/ALL_PHYS_SIGNAL_NoSmearing/*.root"
)

all_files += glob.glob("/workspace/root_files/cnaf/2026-09-06/ALL_PHYS2_SIGNAL_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/cnaf/2026-09-06/ALL_PHYS3_SIGNAL_NoSmearing/*.root")
all_files += glob.glob("/workspace/root_files/cnaf/2026-09-06/DATA_SIGNAL_NoSmearing/*.root")

# Przekaż czystą listę plików bezpośrednio do RDataFrame
df_basic = ROOT.RDataFrame("h1", all_files)

signal_mctruth = 1

mc_signal_flag = f"mcflag == 1 && (mctruth == {signal_mctruth} || mctruth == 0)"
mc_signal_error_flag = f"mcflag == 1 && (mctruth == {signal_mctruth} || mctruth == 0 || mctruth == -1)"
df = df_basic.Define("Kchrec_5", "Kchrec[5]").Define("Knerec_5", "Knerec[5]").Define("MassPi01", "pi01Fit[5]").Define("MassPi02", "pi02Fit[5]").Filter(mc_signal_flag)

df_error = df_basic.Filter(mc_signal_error_flag)

counts = []

counts.append(df.Count())
counts.append(df_error.Count())
counts.append(df_error.Filter(f"mctruth == {signal_mctruth}").Count())

ROOT.RDF.RunGraphs(counts)

for i, count in enumerate(counts):
    counts[i] = counts[i].GetValue()

preselection_eff = counts[0] / counts[1] if counts[1] > 0 else 1.0
selection_eff = counts[2] / counts[0] if counts[0] > 0 else 1.0

print(f"Total count: {counts[1]},\nPreselected: {counts[0]},\nSelected: {counts[2]},\nPreselection efficiency: {preselection_eff * 100:.2f}%,\nSelection efficiency: {selection_eff * 100:.2f}%")

Total count: 954234,
Preselected: 450362,
Selected: 151970,
Preselection efficiency: 47.20%,
Selection efficiency: 33.74%


## Using optimized cuts efficiencies

In [20]:
pi0Mass1_mean = 134.83240924168848
pi0Mass2_mean = 134.87134080668446

# Definicja funkcji w Pythonie (odpowiednik lambd C++)
u = lambda pi01_5, pi02_5: ((pi01_5 - pi0Mass1_mean) + (pi02_5 - pi0Mass2_mean)) / math.sqrt(2)
v = lambda pi01_5, pi02_5: ((pi01_5 - pi0Mass1_mean) - (pi02_5 - pi0Mass2_mean)) / math.sqrt(2)

In [3]:
cuts = {
    "invMassKch": "abs(Kchrec_5 - 497.605) < 2.637",
    "chi2": "Chi2SignalKinFit < 30.0",
    
}

efficiency = []
sel_counts = []
combined_cut = ""

for cut in cuts.values():
    combined_cut += " && (" + cut + ")" if len(combined_cut) > 0 else "(" + cut + ")"
    sel_counts.append(df.Filter(combined_cut).Count())

ROOT.RDF.RunGraphs(sel_counts)

for count in sel_counts:
    count = count.GetValue()
    efficiency.append(count / counts[0] if counts[0] > 0 else 1.0)

    print(f"Count: {count}, Efficiency: {efficiency[-1]*100:.2f}%")

Count: 317400, Efficiency: 70.48%
Count: 194506, Efficiency: 43.19%


In [10]:
histos_chi2 = []
histos_chi2_2D = []

variables = ["Chi2SignalKinFit", "Chi2NeutralKinFit", "Chi2PMKinFit"]
title  = ["#chi^{2}_{+-00} [-]", "#chi^{2}_{00} [-]", "#chi^{2}_{+-} [-]"]
ranges = [[0, 100.0], [0.0, 50.0], [0.0, 5.0]]

for i, var in enumerate(variables):
    histos_chi2.append(df.Filter(combined_cut).Histo1D((f"{var}", f";{title[i]};Counts", 100, ranges[i][0], ranges[i][1]), var))

for i, var in enumerate(variables):
    for j, var2 in enumerate(variables):

        if i >= j:
            continue

        histos_chi2_2D.append(df.Histo2D((f"{var}_{var2}", f";{title[i]};{title[j]}", 100, ranges[i][0], ranges[i][1], 100, ranges[j][0], ranges[j][1]), var, var2))

all_hists = histos_chi2 + histos_chi2_2D

ROOT.RDF.RunGraphs(all_hists)

1

In [17]:
chi2_func = ROOT.TF1("chi2_pdf", "[0] * ROOT::Math::chisquared_pdf(x, [1])", 0, 100)

In [19]:
canvas = ROOT.TCanvas("c1", "", 400, 1200)
canvas.Divide(1,3)

dof = [10, 6, 2]

for i, hist in enumerate(histos_chi2):
    canvas.cd(i + 1)

    chi2_func.SetParameter(0, hist.Integral(0, hist.GetNbinsX() + 1))
    chi2_func.FixParameter(1, dof[i])
    
    hist.Fit(chi2_func, "S", "", 0, 18)
    hist.Draw()

canvas.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      5233.82
NDf                       =           17
Edm                       =  3.70046e-14
NCalls                    =           13
p0                        =       164268   +/-   416.877     
p1                        =           10                      	 (fixed)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      11134.5
NDf                       =           35
Edm                       =  1.23898e-13
NCalls                    =           13
p0                        =      34135.7   +/-   131.051     
p1                        =            6                      	 (fixed)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      18540.5
NDf                       =           99
Edm                       =  5.45998e-11
NCalls                    =           15
p0                        =

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


In [28]:
canvas = ROOT.TCanvas("c1", "", 400, 1200)
canvas.Divide(1,3)

for i, hist in enumerate(histos_chi2_2D):
    canvas.cd(i + 1)
    hist.Draw("COLZ")

canvas.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
